In [0]:
#Load the table
from pyspark.sql import functions as F
from pyspark.sql.window import Window

recommendation_df = spark.table("customer_recommendation")
engagement_df = spark.table("engagement_event")
order_df = spark.table("order_master")

print("Tables loaded successfully!")

In [0]:
#Check the recommendations
display(
    recommendation_df.select(
        "recommendation_id",
        "customer_id",
        "recommended_action",
        "recommendation_date"
    )
)

In [0]:
#Find customer engagement after recommendation
engagement_outcomes_df = (
    recommendation_df.alias("r")
    .join(
        engagement_df.alias("e"),
        (F.col("r.customer_id") == F.col("e.customer_id")) &
        (F.to_date(F.col("e.event_timestamp")) > F.col("r.recommendation_date")) &
        (F.to_date(F.col("e.event_timestamp")) <= F.date_add(F.col("r.recommendation_date"), 30)),
        "left"
    )
    .groupBy(
        F.col("r.recommendation_id"),
        F.col("r.customer_id"),
        F.col("r.recommended_action")
    )
    .agg(
        F.count("e.event_id").alias("engagement_count"),
        F.sum(
            F.when(F.col("e.event_type") == "EMAIL_OPEN", 1).otherwise(0)
        ).alias("email_open_count"),
        F.sum(
            F.when(F.col("e.event_type") == "EMAIL_CLICK", 1).otherwise(0)
        ).alias("email_click_count")
    )
)

display(engagement_outcomes_df)

In [0]:
#Find purchases after recommendation
purchase_outcomes_df = (
    recommendation_df.alias("r")
    .join(
        order_df.alias("o"),
        (F.col("r.customer_id") == F.col("o.customer_id")) &
        (F.col("o.order_status") == "DELIVERED") &
        (F.col("o.order_date") > F.col("r.recommendation_date")) &
        (F.col("o.order_date") <= F.date_add(F.col("r.recommendation_date"), 30)),
        "left"
    )
    .groupBy(
        F.col("r.recommendation_id"),
        F.col("r.customer_id")
    )
    .agg(
        F.count("o.order_id").alias("purchase_count")
    )
)

display(purchase_outcomes_df)

In [0]:
#Combine engagement and purchase outcomes
outcome_df = (
    recommendation_df
    .select(
        "recommendation_id",
        "customer_id",
        "action_id",
        "recommended_action",
        "recommendation_date"
    )
    .join(
        engagement_outcomes_df.select(
            "recommendation_id",
            "engagement_count",
            "email_open_count",
            "email_click_count"
        ),
        "recommendation_id",
        "left"
    )
    .join(
        purchase_outcomes_df.select(
            "recommendation_id",
            "purchase_count"
        ),
        "recommendation_id",
        "left"
    )
    .fillna(0, subset=[
        "engagement_count",
        "email_open_count",
        "email_click_count",
        "purchase_count"
    ])
    .withColumn(
        "outcome_status",
        F.when(F.col("purchase_count") > 0, "CONVERTED")
         .when(F.col("email_click_count") > 0, "CLICKED")
         .when(F.col("email_open_count") > 0, "OPENED")
         .when(F.col("engagement_count") > 0, "ENGAGED")
         .otherwise("NO_RESPONSE")
    )
    .withColumn(
        "outcome_id",
        F.concat(
            F.lit("OUT-2026-"),
            F.lpad(
                F.row_number().over(Window.orderBy("recommendation_id")).cast("string"),
                8,
                "0"
            )
        )
    )
    .withColumn(
        "evaluation_date",
        F.date_add(F.col("recommendation_date"), 30)
    )
)

display(outcome_df)

In [0]:
#Save ActionOutcome
final_outcome_df = outcome_df.select(
    "outcome_id",
    "recommendation_id",
    "customer_id",
    "action_id",
    "recommended_action",
    "recommendation_date",
    "evaluation_date",
    "engagement_count",
    "email_open_count",
    "email_click_count",
    "purchase_count",
    "outcome_status"
)

final_outcome_df.write.format("delta").mode("overwrite").saveAsTable(
    "action_outcome"
)

print("Action Outcome table created successfully!")

In [0]:
#Analyze the results
final_outcome_df.groupBy(
    "outcome_status"
).count().orderBy("outcome_status").show()

In [0]:
#Calculate the conversion rate:
total_recommendations = final_outcome_df.count()

converted_customers = final_outcome_df.filter(
    F.col("outcome_status") == "CONVERTED"
).count()

conversion_rate = (
    converted_customers / total_recommendations * 100
    if total_recommendations > 0 else 0
)

print("Total recommendations:", total_recommendations)
print("Converted recommendations:", converted_customers)
print("Conversion rate:", round(conversion_rate, 2), "%")